In [1]:
# df.csvとdf_test.csvを綺麗に
import pandas as pd
import re
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

In [2]:
# csvをデータフレームへ
df1 = pd.read_csv('df.csv')
df2 = pd.read_csv('df_test.csv')

In [3]:
# 不要な列を削除
df1.drop(['賃料', '所在地', '所在階', 'アクセス', '間取り', '築年数', '方角', '面積', '所在階', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '建物構造', '契約期間'], axis=1, inplace=True)
df2.drop(['所在地', '所在階', 'アクセス', '間取り', '築年数', '方角', '面積', '所在階', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '建物構造', '契約期間'], axis=1, inplace=True)

In [4]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31470 entries, 0 to 31469
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      31470 non-null  int64  
 1   整形住所    31470 non-null  object 
 2   緯度      30313 non-null  float64
 3   経度      30313 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 983.6+ KB


In [5]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31262 entries, 0 to 31261
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      31262 non-null  int64  
 1   整形住所    31262 non-null  object 
 2   緯度      30149 non-null  float64
 3   経度      30149 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 977.1+ KB


In [6]:
#df1.to_csv('geocode_train.csv', index=False)
#df2.to_csv('geocode_test.csv', index=False)

In [7]:
# csvをデータフレームへ
df3 = pd.read_csv('geocode_train.csv')
df4 = pd.read_csv('geocode_test.csv')

In [8]:
# Geolocatorのインスタンスを作成
geolocator = Nominatim(user_agent="your_app_name")

In [9]:
def get_lat_long(location):
    retries = 3  # 最大リトライ回数
    for attempt in range(retries):
        try:
            location = geolocator.geocode(location, timeout=10)  # タイムアウトを10秒に設定
            if location:
                return location.latitude, location.longitude
            return None, None
        except GeocoderTimedOut:
            if attempt < retries - 1:  # 最後の試行でない場合
                time.sleep(2)  # 2秒待機してリトライ
            else:
                print(f"Geocoder service timed out for {location}.")
                return None, None

In [10]:
# 緯度がブランクの行にのみジオコーディングを実行
for idx, row in df3[df3['緯度'].isna()].iterrows():
    lat, lon = get_lat_long(row['整形住所'])
    df3.at[idx, '緯度'] = lat
    df3.at[idx, '経度'] = lon
    time.sleep(1)  # リクエスト間で1秒スリープ

In [11]:
# 緯度がブランクの行にのみジオコーディングを実行
for idx, row in df4[df4['緯度'].isna()].iterrows():
    lat, lon = get_lat_long(row['整形住所'])
    df4.at[idx, '緯度'] = lat
    df4.at[idx, '経度'] = lon
    time.sleep(1)  # リクエスト間で1秒スリープ

In [12]:
df3.to_csv('geocode_train2.csv', index=False)
df4.to_csv('geocode_test2.csv', index=False)

In [1]:
import pandas as pd

In [2]:
# csvをデータフレームへ
df3 = pd.read_csv('geocode_train2.csv')
df4 = pd.read_csv('geocode_test2.csv')

In [3]:
# 東京都23区のリスト
wards = [
    "千代田区", "中央区", "港区", "新宿区", "文京区", "台東区", "墨田区", "江東区",
    "品川区", "目黒区", "大田区", "世田谷区", "渋谷区", "中野区", "杉並区", "豊島区",
    "北区", "荒川区", "板橋区", "練馬区", "足立区", "葛飾区", "江戸川区"
]

In [4]:
# 正規表現を使って「何区」を抽出する関数
def extract_ward(address):
    # 23区リストを元に住所から該当する区を検索
    for ward in wards:
        if ward in address:
            return ward
    return None  # 該当する区がなければ None を返す

In [5]:
# 住所データを含む列に対して処理を適用
df3['区'] = df3['整形住所'].apply(extract_ward)
df4['区'] = df4['整形住所'].apply(extract_ward)

In [6]:
# 区ごとの緯度・経度の平均を計算
mean_lat_long = df3.groupby('区')[['緯度', '経度']].mean()

In [7]:
# 欠損値を区ごとの平均値で補完する関数
def fill_missing_lat_long(row):
    if pd.isnull(row['緯度']) or pd.isnull(row['経度']):
        # 区ごとの平均値を取得
        mean_values = mean_lat_long.loc[row['区']]
        if pd.isnull(row['緯度']):
            row['緯度'] = mean_values['緯度']
        if pd.isnull(row['経度']):
            row['経度'] = mean_values['経度']
    return row

In [9]:
# 欠損値を補完
df3 = df3.apply(fill_missing_lat_long, axis=1)
df4 = df4.apply(fill_missing_lat_long, axis=1)

In [10]:
df3.to_csv('geocode_train.csv', index=False)
df4.to_csv('geocode_test.csv', index=False)